## Made by Бутер Бродский aka Nyamerka)

### Воропаев Иван, М8О-309Б-22

In [512]:
# Импортируем тонны библиотек
import os
import optuna
import logging
from optuna.samplers import TPESampler
from boruta import BorutaPy
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import make_scorer, roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings
from catboost import CatBoostClassifier
import category_encoders as ce
warnings.filterwarnings("ignore")
logging.getLogger('optuna').setLevel(logging.WARNING)

Для начала нужно считать данные:

In [513]:
loan_data = pd.read_csv('train.csv')

Отлично. Теперь у нас есть данные - начинаем обработку.

In [514]:
loan_data.head()
info = loan_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11017 entries, 0 to 11016
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ApplicationDate             10487 non-null  object 
 1   Age                         10487 non-null  float64
 2   AnnualIncome                10487 non-null  float64
 3   CreditScore                 9986 non-null   float64
 4   LoanAmount                  9986 non-null   float64
 5   LoanDuration                10487 non-null  float64
 6   MaritalStatus               10487 non-null  object 
 7   NumberOfDependents          10487 non-null  float64
 8   HomeOwnershipStatus         10487 non-null  object 
 9   MonthlyDebtPayments         9986 non-null   float64
 10  CreditCardUtilizationRate   10487 non-null  float64
 11  NumberOfOpenCreditLines     10487 non-null  float64
 12  NumberOfCreditInquiries     10487 non-null  float64
 13  DebtToIncomeRatio           104

In [515]:
loan_data.describe()

,Age,AnnualIncome,CreditScore,LoanAmount,LoanDuration,NumberOfDependents,MonthlyDebtPayments,CreditCardUtilizationRate,NumberOfOpenCreditLines,NumberOfCreditInquiries,...,UtilityBillsPaymentHistory,JobTenure,Experience,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,LoanApproved,RiskScore
count,10487.000000,10487.000000,9986.000000,9986.000000,10487.000000,10487.000000,9986.000000,10487.000000,10487.000000,10487.000000,...,10487.000000,10487.000000,10487.000000,9.986000e+03,9986.000000,10487.000000,10487.000000,10487.000000,10487.000000,1.048700e+04
mean,39.850386,131587.872127,678.082716,29874.218306,53.439878,1.568323,546.458642,0.284397,3.033565,0.979498,...,0.784428,4.949271,17.628302,1.542381e+05,0.200392,0.200112,1075.622426,0.517577,0.511776,-2.569878e+04
std,11.614132,115791.941909,175.192486,27705.509722,24.493562,1.418684,501.981888,0.159240,1.740186,0.990927,...,0.123039,2.201100,11.337248,4.622229e+05,0.094388,0.096458,1344.053181,0.894637,0.499885,1.431675e+06
min,18.000000,15000.000000,300.000000,1063.000000,12.000000,0.000000,13.000000,0.003674,0.000000,0.000000,...,0.259301,0.000000,0.000000,1.004000e+03,0.052494,0.046445,30.008506,0.006064,0.000000,-9.999999e+06
25%,32.000000,20959.500000,550.000000,12658.000000,36.000000,0.000000,233.250000,0.158929,2.000000,0.000000,...,0.708475,3.000000,9.000000,7.252500e+03,0.119908,0.119548,375.872620,0.066734,0.000000,3.256475e+01
50%,40.000000,89015.000000,722.500000,21828.500000,48.000000,1.000000,398.000000,0.262229,3.000000,1.000000,...,0.803692,5.000000,17.000000,2.742950e+04,0.182023,0.180710,684.878529,0.178193,1.000000,4.411876e+01
75%,48.000000,257025.000000,850.000000,37158.000000,60.000000,3.000000,685.000000,0.391683,4.000000,2.000000,...,0.879312,6.000000,26.000000,1.241758e+05,0.264709,0.264880,1279.930203,0.637457,1.000000,6.535690e+01
max,80.000000,748508.000000,850.000000,418997.000000,120.000000,6.000000,10879.000000,0.914635,12.000000,6.000000,...,0.996573,17.000000,57.000000,1.126117e+07,0.722497,0.833647,29634.807816,24.383046,1.000000,1.000000e+07


In [516]:
loan_data.isnull().sum()

ApplicationDate                530
Age                            530
AnnualIncome                   530
CreditScore                   1031
LoanAmount                    1031
LoanDuration                   530
MaritalStatus                  530
NumberOfDependents             530
HomeOwnershipStatus            530
MonthlyDebtPayments           1031
CreditCardUtilizationRate      530
NumberOfOpenCreditLines        530
NumberOfCreditInquiries        530
DebtToIncomeRatio              530
BankruptcyHistory             1031
LoanPurpose                   1031
PreviousLoanDefaults           530
PaymentHistory                 530
LengthOfCreditHistory          530
SavingsAccountBalance          530
CheckingAccountBalance        1031
TotalAssets                   1031
TotalLiabilities               530
MonthlyIncome                  530
UtilityBillsPaymentHistory     530
JobTenure                      530
EmploymentStatus               530
EducationLevel                 530
Experience          

In [517]:
loan_data.duplicated().sum()

1016

In [518]:
print(loan_data.nunique(), loan_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11017 entries, 0 to 11016
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ApplicationDate             10487 non-null  object 
 1   Age                         10487 non-null  float64
 2   AnnualIncome                10487 non-null  float64
 3   CreditScore                 9986 non-null   float64
 4   LoanAmount                  9986 non-null   float64
 5   LoanDuration                10487 non-null  float64
 6   MaritalStatus               10487 non-null  object 
 7   NumberOfDependents          10487 non-null  float64
 8   HomeOwnershipStatus         10487 non-null  object 
 9   MonthlyDebtPayments         9986 non-null   float64
 10  CreditCardUtilizationRate   10487 non-null  float64
 11  NumberOfOpenCreditLines     10487 non-null  float64
 12  NumberOfCreditInquiries     10487 non-null  float64
 13  DebtToIncomeRatio           104

In [519]:
loan_data = loan_data.dropna()
loan_data = loan_data.loc[(loan_data['RiskScore'] >= 0) & (loan_data['RiskScore'] <= 100)]
loan_data.drop_duplicates(inplace=True)
print(loan_data['RiskScore'].min(), loan_data['RiskScore'].max())
print(set(loan_data['LoanApproved'])) # Должны быть только значения 0 и 1 - то есть да или нет.

14.841417296887238 97.59724939432462
{0.0, 1.0}


По идее, на данном этапе должна начаться предобработка признаков. Однако у нас есть некоторый выбор алгоритма бустинга. По идее, это либо XGBoost, либо CatBoost. Однако у них есть принципиальное отличие.

+ CatBoost - поддерживает обработку категорий. То есть предобработка не нужна.
+ XGBoost - не поддерживает категории, то есть они уже должны быть обработаны.
+ LightBoost - также без категорий, но быстро.

В общем, используем Cat

Дату нужно будет распарсить в любом случае)

In [520]:
loan_data['ApplicationDate'] = pd.to_datetime(loan_data['ApplicationDate'])

loan_data['ApplicationYear'] = loan_data['ApplicationDate'].dt.year
loan_data['ApplicationMonth'] = loan_data['ApplicationDate'].dt.month
loan_data['ApplicationDay'] = loan_data['ApplicationDate'].dt.day

loan_data = loan_data.drop(columns=['ApplicationDate'])

Разбиваем сет на части

In [521]:
categorical_columns = ['MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose','EmploymentStatus', 'EducationLevel', 'ApplicationYear', 'ApplicationMonth', 'ApplicationDay']

target = 'LoanApproved'

numerical_columns = loan_data.drop(columns=categorical_columns + [target]).columns

print(numerical_columns)

if len(numerical_columns) + len(categorical_columns) + len([target]) != len(loan_data.columns):
    raise ValueError('Количество колонок не совпадает')

Index(['Age', 'AnnualIncome', 'CreditScore', 'LoanAmount', 'LoanDuration',
       'NumberOfDependents', 'MonthlyDebtPayments',
       'CreditCardUtilizationRate', 'NumberOfOpenCreditLines',
       'NumberOfCreditInquiries', 'DebtToIncomeRatio', 'BankruptcyHistory',
       'PreviousLoanDefaults', 'PaymentHistory', 'LengthOfCreditHistory',
       'SavingsAccountBalance', 'CheckingAccountBalance', 'TotalAssets',
       'TotalLiabilities', 'MonthlyIncome', 'UtilityBillsPaymentHistory',
       'JobTenure', 'Experience', 'NetWorth', 'BaseInterestRate',
       'InterestRate', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio',
       'RiskScore'],
      dtype='object')


Строим графики

In [ ]:
from matplotlib import pyplot as plt

loan_data.hist(color='Green', bins=20, figsize=(25,30))
plt.suptitle('Распределения данных по каждому столбцу', fontsize=20)
plt.show()

Добавляем HeatMap

In [ ]:
import seaborn as sns

plt.figure(figsize=(30, 20))
plt.title('Корреляция численных данных')
loan_data.corr(numeric_only=True)
sns.heatmap(loan_data.corr(numeric_only=True), annot=True, fmt=".2f")

Создаём новые фичи. Квадтары каждой численной метрики и перемноженные друг на друга признаки.

In [522]:
def create_new_features(loan_data, numerical_columns, degree=2):
    for col1, col2 in combinations(numerical_columns, 2):
        new_col_name = f"{col1}_x_{col2}"
        loan_data[new_col_name] = loan_data[col1] * loan_data[col2]

    poly = PolynomialFeatures(degree=degree, include_bias=False)
    poly_features = poly.fit_transform(loan_data[numerical_columns])

    poly_feature_names = poly.get_feature_names_out(numerical_columns)

    poly_df = pd.DataFrame(poly_features, columns=poly_feature_names, index=loan_data.index)
    loan_data = pd.concat([loan_data, poly_df], axis=1)

    loan_data = loan_data.loc[:, ~loan_data.columns.duplicated()]
    return loan_data

print(numerical_columns)

loan_data = create_new_features(loan_data, numerical_columns)

print(f"Размер данных после генерации новых признаков: {loan_data.shape}")

Index(['Age', 'AnnualIncome', 'CreditScore', 'LoanAmount', 'LoanDuration',
       'NumberOfDependents', 'MonthlyDebtPayments',
       'CreditCardUtilizationRate', 'NumberOfOpenCreditLines',
       'NumberOfCreditInquiries', 'DebtToIncomeRatio', 'BankruptcyHistory',
       'PreviousLoanDefaults', 'PaymentHistory', 'LengthOfCreditHistory',
       'SavingsAccountBalance', 'CheckingAccountBalance', 'TotalAssets',
       'TotalLiabilities', 'MonthlyIncome', 'UtilityBillsPaymentHistory',
       'JobTenure', 'Experience', 'NetWorth', 'BaseInterestRate',
       'InterestRate', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio',
       'RiskScore'],
      dtype='object')
Размер данных после генерации новых признаков: (9332, 879)


Получили 879 признаков. Нужно обновить наши колонки, так как появились новые.

In [523]:
# Пересчитываем списки колонок
def update_columns(loan_data, target, predefined_categorical):
    # Все категориальные данные, которые остались неизменными
    categorical_columns = [col for col in predefined_categorical if col in loan_data.columns]

    # Числовые колонки определяем как все оставшиеся, кроме целевой переменной и категориальных
    numerical_columns = list(set(loan_data.columns) - set(categorical_columns) - set([target]))

    return numerical_columns, categorical_columns

# Обновляем списки колонок
numerical_columns, categorical_columns = update_columns(loan_data, target, categorical_columns)

# Проверка корректности
if len(numerical_columns) + len(categorical_columns) + 1 != len(loan_data.columns):
    raise ValueError('Количество колонок не совпадает')

# Выводим размеры и актуальные списки
print(f"Размер данных: {loan_data.shape}")
print(f"Числовые колонки: {numerical_columns}")
print(f"Категориальные колонки: {categorical_columns}")

Размер данных: (9332, 879)
Числовые колонки: ['JobTenure_x_TotalDebtToIncomeRatio', 'MonthlyDebtPayments_x_CreditCardUtilizationRate', 'NumberOfCreditInquiries_x_MonthlyIncome', 'LoanAmount_x_MonthlyLoanPayment', 'CreditCardUtilizationRate BankruptcyHistory', 'MonthlyDebtPayments_x_TotalLiabilities', 'CreditCardUtilizationRate_x_TotalLiabilities', 'PreviousLoanDefaults TotalAssets', 'CreditCardUtilizationRate NumberOfOpenCreditLines', 'LoanAmount TotalLiabilities', 'NetWorth InterestRate', 'BaseInterestRate RiskScore', 'UtilityBillsPaymentHistory JobTenure', 'LengthOfCreditHistory_x_TotalLiabilities', 'NumberOfCreditInquiries_x_BaseInterestRate', 'PreviousLoanDefaults_x_TotalDebtToIncomeRatio', 'PaymentHistory_x_UtilityBillsPaymentHistory', 'NumberOfDependents LengthOfCreditHistory', 'NumberOfDependents UtilityBillsPaymentHistory', 'PreviousLoanDefaults LengthOfCreditHistory', 'BankruptcyHistory UtilityBillsPaymentHistory', 'BankruptcyHistory Experience', 'TotalAssets RiskScore', 'Inte

Кодируем данные енкодером из CatBoosting.

In [524]:
# Обновляем списки колонок после добавления новых фичей
def encode_categorical_features(loan_data, categorical_columns, target):
    if not categorical_columns:
        print("Категориальные признаки отсутствуют. Пропускаем обработку.")
        return loan_data

    # Инициализация TargetEncoder
    encoder = ce.TargetEncoder(cols=categorical_columns)

    # Преобразуем данные
    loan_data[categorical_columns] = encoder.fit_transform(loan_data[categorical_columns], loan_data[target])

    return loan_data

# Применяем обработку категориальных признаков
loan_data = encode_categorical_features(loan_data, categorical_columns, target)

# Проверяем и преобразуем оставшиеся object-столбцы
for col in categorical_columns:
    if loan_data[col].dtype == 'object' or loan_data[col].dtype == 'category':
        print(f"Преобразуем колонку {col} из object в числовой формат.")
        # Посмотрим на уникальные значения
        print(f"Уникальные значения в колонке {col}:", loan_data[col].unique())

        # Преобразуем в числовой формат, заменяя некорректные значения на NaN
        loan_data[col] = pd.to_numeric(loan_data[col], errors='coerce')

        # Если остались NaN, заполним их средним значением
        if loan_data[col].isna().sum() > 0:
            print(f"Найдены NaN в колонке {col}, заполняем средним значением.")
            loan_data[col] = loan_data[col].fillna(loan_data[col].mean())

# Принудительное преобразование колонок в float
columns_to_convert = ['UtilityBillsPaymentHistory', 'TotalDebtToIncomeRatio']

for col in columns_to_convert:
    print(f"Преобразуем колонку {col} в float.")
    loan_data[col] = pd.to_numeric(loan_data[col], errors='coerce')

# Проверяем, что преобразование прошло успешно
unsupported_types = loan_data.select_dtypes(include=['object']).columns.tolist()
if unsupported_types:
    raise ValueError(f"После преобразования остаются неподдерживаемые типы: {unsupported_types}")

# Проверка результата
print(f"Размер данных после обработки категорий: {loan_data.shape}")

# Проверяем, что все столбцы имеют поддерживаемые XGBoost типы
unsupported_types = loan_data.select_dtypes(include=['object']).columns.tolist()
if unsupported_types:
    raise ValueError(f"Обнаружены неподдерживаемые типы: {unsupported_types}")

if len(numerical_columns) + len(categorical_columns) + 1 != len(loan_data.columns):
    raise ValueError('Количество колонок не совпадает')

Преобразуем колонку UtilityBillsPaymentHistory в float.
Преобразуем колонку TotalDebtToIncomeRatio в float.
Размер данных после обработки категорий: (9332, 879)


Выбираем, какой Scaler лучше всего использовать.

In [525]:
loan_data[categorical_columns] = loan_data[categorical_columns].astype(str)

# Функция для выбора и применения скейлера
def normalize_and_evaluate(trial, loan_data, numerical_columns, target):
    # Выбор скейлера
    scaler_name = trial.suggest_categorical('scaler', ['StandardScaler', 'MinMaxScaler', 'RobustScaler', 'MaxAbsScaler'])

    if scaler_name == 'StandardScaler':
        scaler = StandardScaler()
    elif scaler_name == 'MinMaxScaler':
        scaler = MinMaxScaler()
    elif scaler_name == 'RobustScaler':
        scaler = RobustScaler()
    elif scaler_name == 'MaxAbsScaler':
        scaler = MaxAbsScaler()
    else:
        raise ValueError("Unknown scaler selected.")

    # Применение скейлера только к числовым данным
    loan_data[numerical_columns] = scaler.fit_transform(loan_data[numerical_columns])

    # Разделение данных
    X = loan_data.drop(columns=[target])
    y = loan_data[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Обучение модели
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    # Оценка модели
    y_pred = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    trial.set_user_attr("scaler", scaler)  # Сохраняем скейлер в атрибутах
    return auc

# Функция оптимизации скейлера
def optimize_scaler(loan_data, numerical_columns, categorical_columns, target, n_trials=20):
    def objective(trial):
        return normalize_and_evaluate(trial, loan_data.copy(), numerical_columns, target)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    # Сохранение лучшего скейлера
    best_trial = study.best_trial
    best_scaler = best_trial.user_attrs["scaler"]
    return best_scaler

# Оптимизация скейлера
best_scaler = optimize_scaler(loan_data, numerical_columns, categorical_columns, target, n_trials=20)

# Применение лучшего скейлера к числовым данным
loan_data[numerical_columns] = best_scaler.fit_transform(loan_data[numerical_columns])

print(best_scaler)

# Проверка итогового размера данных
print(f"Размер данных после нормализации: {loan_data.shape}")

MaxAbsScaler()
Размер данных после нормализации: (9332, 879)


Начинаем отбор признаков с помощью Boruta.

In [526]:
features_file = "selected_features.txt"

if os.path.exists(features_file):
    print("Файл с отобранными признаками найден. Загружаем признаки...")
    # Читаем признаки из файла
    with open(features_file, "r") as f:
        selected_features = f.read().splitlines()
    print("Загруженные признаки:", selected_features)
else:
    print("Файл с отобранными признаками не найден. Запускаем Boruta...")
    # Данные для Boruta
    X_class = loan_data.drop(columns=target)
    y_class = loan_data[target]

    # Оценка с использованием ROC AUC
    scorer = make_scorer(roc_auc_score, needs_proba=True)

    # Random Forest для Boruta
    rf = RandomForestClassifier(n_estimators=300, random_state=42)

    # Boruta
    boruta = BorutaPy(estimator=rf, n_estimators='auto', random_state=42)
    boruta.fit(X_class.values, y_class.values)

    # Список выбранных признаков
    selected_features = X_class.columns[boruta.support_].tolist()
    print("Выбранные признаки Boruta:", selected_features)

    # Сохраняем признаки в файл
    with open(features_file, "w") as f:
        f.write("\n".join(selected_features))
    print("Признаки сохранены в файл.")

# Выбранные признаки можно использовать для дальнейшего анализа
print(f"Итоговый список признаков: {selected_features}")

Файл с отобранными признаками не найден. Запускаем Boruta...
Выбранные признаки Boruta: ['AnnualIncome', 'CreditScore', 'LoanAmount', 'PaymentHistory', 'TotalAssets', 'MonthlyIncome', 'UtilityBillsPaymentHistory', 'NetWorth', 'BaseInterestRate', 'InterestRate', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio', 'RiskScore', 'Age_x_AnnualIncome', 'Age_x_LoanAmount', 'Age_x_TotalAssets', 'Age_x_MonthlyIncome', 'Age_x_NetWorth', 'Age_x_BaseInterestRate', 'Age_x_TotalDebtToIncomeRatio', 'Age_x_RiskScore', 'AnnualIncome_x_CreditScore', 'AnnualIncome_x_LoanAmount', 'AnnualIncome_x_LoanDuration', 'AnnualIncome_x_MonthlyDebtPayments', 'AnnualIncome_x_CreditCardUtilizationRate', 'AnnualIncome_x_NumberOfOpenCreditLines', 'AnnualIncome_x_DebtToIncomeRatio', 'AnnualIncome_x_PaymentHistory', 'AnnualIncome_x_LengthOfCreditHistory', 'AnnualIncome_x_TotalAssets', 'AnnualIncome_x_MonthlyIncome', 'AnnualIncome_x_UtilityBillsPaymentHistory', 'AnnualIncome_x_JobTenure', 'AnnualIncome_x_Experience', 'AnnualIn

In [527]:
# print(selected_features)

Обучаем бустинг с помощью Optuna и выводим результат.

In [530]:
# Фильтрация признаков
X_class_selected = loan_data[selected_features]
y_class = loan_data[target]

# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X_class_selected, y_class, test_size=0.4, random_state=42)

# Оптимизация гиперпараметров с помощью Optuna
def objective(trial):
    # Выбор модели
    model_name = trial.suggest_categorical("model", ["catboost"])

    if model_name == "catboost":
        params = {
            "iterations": trial.suggest_int("iterations", 50, 500),
            "depth": trial.suggest_int("depth", 1, 3),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
            "random_seed": 42,
            "verbose": 0,
            "early_stopping_rounds": 50
        }
        model = CatBoostClassifier(**params)

    # Кросс-валидация
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1", n_jobs=-1)
    return np.mean(scores)

# Настройка Optuna
study = optuna.create_study(direction="maximize", sampler=TPESampler())
study.optimize(objective, n_trials=50)

# Лучшие параметры
best_params = study.best_params
best_model_name = best_params.pop("model")
print(f"Лучшая модель: {best_model_name}")
print(f"Лучшие параметры: {best_params}")

# Финальное обучение
if best_model_name == "catboost":
    final_model = CatBoostClassifier(**best_params, random_seed=42, verbose=0)

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)  # Предсказание классов (0 или 1)
final_f1 = f1_score(y_test, y_pred)
print(f"Итоговый f1: {final_f1}")

Лучшая модель: catboost
Лучшие параметры: {'iterations': 214, 'depth': 3, 'learning_rate': 0.00725317892349884, 'l2_leaf_reg': 3.1843859023245287}
Итоговый f1: 0.9843912591050988


In [531]:
print("Directed by Бутер Бродский")

Directed by Бутер Бродский
